In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Celebal Assignment 05") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.2.0


In [3]:
df = spark.read.csv(
    "../data/Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [4]:
# Display schema
df.printSchema()

# Total rows and columns
print("Rows:", df.count())
print("Columns:", len(df.columns))

# Column names
print(df.columns)

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)

Rows: 9994
Columns: 21
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City

In [5]:
from pyspark.sql.functions import col, when, count

# Count null values in each column
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [6]:
print("Rows before:", df.count())

df = df.dropDuplicates()

print("Rows after:", df.count())

Rows before: 9994
Rows after: 9994


In [7]:
# Fill missing values

df = df.fillna({
    "Postal Code": 0,
    "Sales": 0,
    "Profit": 0,
    "Quantity": 0
})

print("Missing values handled successfully.")

Missing values handled successfully.


In [ ]:
## Filtering Operations

The dataset was filtered using different business conditions to retrieve meaningful subsets of data. These include sales amount, product category, region, discount, profit, and multiple combined conditions.

In [5]:
from pyspark.sql.functions import col

df.filter(col("Sales") > 500).show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|    Segment|      Country|           City|       State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|    Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|     2|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute|   Consumer|United States|      Henderson|    Kentucky|      42420|  South|FUR-CH-10000454|      Furniture| 

In [6]:
df.filter(col("Category") == "Furniture").show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+------------+-----------+-------+---------------+---------+------------+--------------------+--------+--------+--------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|       State|Postal Code| Region|     Product ID| Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|    Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+------------+-----------+-------+---------------+---------+------------+--------------------+--------+--------+--------+----------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|    Kentucky|      42420|  South|FUR-BO-10001798|Furniture|   Bookcases|Bush Somerset Col...

In [8]:
df.filter(col("Region") == "West").show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|       City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     3|CA-2016-138688| 6/12/2016| 6/16/2016|  Second Class|   DV-13045|Darrin Van Huff|Corporate|United States|Los Angeles|California|      90036|  West|OFF-LA-10000240|Office Supplies|      Labels|Self-Adhesive Add...|   14.62|

In [7]:
df.filter(col("Profit") > 100).show(10)

+------+--------------+----------+----------+--------------+-----------+--------------+---------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID| Customer Name|  Segment|      Country|         City|     State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+--------------+---------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     2|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|   Claire Gute| Consumer|United States|    Henderson|  Kentucky|      42420|  South|FUR-CH-10000454|      Furniture|      Chairs|Hon Deluxe Fabric...| 

In [9]:
df.filter(
    (col("Region") == "West") &
    (col("Category") == "Technology")
).show(10)

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+-------------+----------+-----------+------+---------------+----------+------------+--------------------+-------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|         City|     State|Postal Code|Region|     Product ID|  Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+-------------+----------+-----------+------+---------------+----------+------------+--------------------+-------+--------+--------+--------+
|     8|CA-2014-115812|  6/9/2014| 6/14/2014|Standard Class|   BH-11710|   Brosina Hoffman|   Consumer|United States|  Los Angeles|California|      90032|  West|TEC-PH-10002275|Technology|      Phones|Mitel 5320 IP Pho...|907.

In [10]:
df.filter(col("Discount") > 0.2).show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|    Segment|      Country|           City|       State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|    Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|     4|US-2015-108966|10/11/2015|10/18/2015|Standard Class|   SO-20335| Sean O'Donnell|   Consumer|United States|Fort Lauderdale|     Florida|      33311|  South|FUR-TA-10000577|      Furniture| 

In [13]:
# Calculate overall sales statistics using aggregate functions.

from pyspark.sql.functions import count, sum, avg, min, max

df.select(
    count("*").alias("Total Records"),
    sum("Sales").alias("Total Sales"),
    avg("Sales").alias("Average Sales"),
    min("Sales").alias("Minimum Sales"),
    max("Sales").alias("Maximum Sales")
).show()

+-------------+------------------+------------------+-------------+-------------+
|Total Records|       Total Sales|     Average Sales|Minimum Sales|Maximum Sales|
+-------------+------------------+------------------+-------------+-------------+
|         9994|2272449.8562999545|234.41818199917006|        0.444|     22638.48|
+-------------+------------------+------------------+-------------+-------------+



In [14]:
# Summarize sales, profit, and orders for each product category.

from pyspark.sql.functions import sum, avg, count

df.groupBy("Category").agg(
    sum("Sales").alias("Total Sales"),
    avg("Profit").alias("Average Profit"),
    count("*").alias("Total Orders")
).show()

+---------------+-----------------+------------------+------------+
|       Category|      Total Sales|    Average Profit|Total Orders|
+---------------+-----------------+------------------+------------+
|Office Supplies|703502.9280000031|20.018731895121128|        6026|
|      Furniture|733046.8612999996| 9.281672418670453|        2121|
|     Technology|835900.0669999964| 78.71591586356247|        1847|
+---------------+-----------------+------------------+------------+



In [15]:
# Display regions with total sales greater than 100000.

from pyspark.sql.functions import col, sum

df.groupBy("Region") \
  .agg(sum("Sales").alias("Total Sales")) \
  .filter(col("Total Sales") > 100000) \
  .show()

+-------+------------------+
| Region|       Total Sales|
+-------+------------------+
|  South|388983.58500000037|
|Central| 497800.8728000007|
|   East| 672194.0539999981|
|   West| 713471.3445000004|
+-------+------------------+



In [16]:
# Rename the Customer Name column.

df = df.withColumnRenamed("Customer Name", "Customer_Name")

df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [17]:
# Convert Quantity and Discount to numeric data types.

from pyspark.sql.functions import col

df = df.withColumn("Quantity", col("Quantity").cast("int")) \
       .withColumn("Discount", col("Discount").cast("double"))

df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [18]:
# Build a complete Spark data processing pipeline.

from pyspark.sql.functions import col, sum, avg, count

pipeline_df = (
    df.dropDuplicates()
      .fillna({
          "Sales": 0,
          "Profit": 0,
          "Quantity": 0,
          "Discount": 0
      })
      .filter(col("Sales") > 100)
)

pipeline_result = (
    pipeline_df
    .groupBy("Category", "Region")
    .agg(
        count("*").alias("Total_Orders"),
        sum("Sales").alias("Total_Sales"),
        avg("Profit").alias("Average_Profit")
    )
    .orderBy(col("Total_Sales").desc())
)

pipeline_result.show()

+---------------+-------+------------+------------------+------------------+
|       Category| Region|Total_Orders|       Total_Sales|    Average_Profit|
+---------------+-------+------------+------------------+------------------+
|     Technology|   East|         336|254948.83500000017|136.33589047619049|
|     Technology|   West|         400|241182.05499999996|106.75313225000004|
|      Furniture|   West|         444|238689.57050000047| 20.33413806306306|
|      Furniture|   East|         363| 196345.1120000001|2.0867289256198394|
|Office Supplies|   West|         383|171408.38300000012|100.83387545691913|
|Office Supplies|   East|         350|166147.95799999996| 91.03385542857143|
|     Technology|Central|         270|        163407.818| 120.5929837037038|
|      Furniture|Central|         285|156368.16040000002|-2.469209473684213|
|     Technology|  South|         181| 143151.1719999999|104.61997513812162|
|Office Supplies|Central|         270|136892.50300000006| 37.78368888888888|

In [ ]:
## Wide Transformation

The `groupBy()` operation is a wide transformation because Spark redistributes (shuffles) data across partitions before performing aggregation. This enables efficient parallel processing of grouped data but involves network communication between partitions.

In [ ]:
# Insights

- Spark DataFrames provide efficient in-memory data processing compared to traditional MapReduce.
- Duplicate records and missing values were handled to improve data quality.
- Business filters helped analyze sales by category, region, profit, and discount.
- Aggregation functions summarized key sales metrics effectively.
- GroupBy operations provided category-wise and region-wise business insights.
- Schema modification ensured correct data types for accurate analysis.
- The complete pipeline demonstrated an end-to-end Spark workflow from data cleaning to aggregation.